# Phase 2 EDA — PSI collector sanity checks

Inputs: one or more `*.labeled.csv` files produced by `collector.py` then `label.py`.
Set `LABELED_CSVS` below before running.

In [ ]:
# Cell 1 — load CSV(s). Uses pandas for convenience (see requirements.txt).
from pathlib import Path
import pandas as pd

LABELED_CSVS = sorted(Path('data').glob('*.labeled.csv'))
assert LABELED_CSVS, 'No labeled CSVs under research/data/ — run collector.py + label.py first.'

df = pd.concat([pd.read_csv(p) for p in LABELED_CSVS], ignore_index=True)
print(f'loaded {len(df)} rows from {len(LABELED_CSVS)} files')
print(df.dtypes)
df.head()

In [ ]:
# Cell 2 — class balance.
pos = int((df['kill_event'] == 1).sum())
neg = int((df['kill_event'] == 0).sum())
pos_fraction = pos / (pos + neg) if (pos + neg) else 0.0
print(f'positives={pos}  negatives={neg}  pos_fraction={pos_fraction:.4f}')
print('per-scenario counts:')
print(df.groupby(['scenario', 'kill_event']).size().unstack(fill_value=0))

In [ ]:
# Cell 3 — some_avg10 distribution per scenario.
import matplotlib.pyplot as plt

scenarios = sorted(df['scenario'].dropna().unique())
fig, ax = plt.subplots(figsize=(8, 4))
for s in scenarios:
    sub = df.loc[df['scenario'] == s, 'some_avg10'].dropna()
    ax.hist(sub, bins=60, alpha=0.45, label=s)
ax.set_xlabel('some_avg10 (%)')
ax.set_ylabel('samples')
ax.set_title('PSI some_avg10 distribution per scenario')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 4 — Phase 2 verification gate from plan-executable.md.
assert 0.005 <= pos_fraction <= 0.05, (
    f'positive fraction {pos_fraction:.4f} outside [0.005, 0.05] — '
    'revisit label window or workload selection'
)
print('OK — positive fraction within Phase 2 spec.')